In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, shutil
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)

for f in ['.gitconfig', '.git-credentials']:
    src = f'/content/drive/MyDrive/XIDS_Research/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/root/{f}')
        if f == '.git-credentials':
            os.chmod(f'/root/{f}', 0o600)

print(f'Ready in: {os.getcwd()}')


In [ ]:
import numpy as np, pandas as pd, json, time, itertools
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')
from scipy.stats import beta as BetaDist, binom
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

TABLES = Path(REPO) / 'results' / 'tables'
FIGS = Path(REPO) / 'results' / 'figures'
DOCS = Path(REPO) / 'docs'
DOCS.mkdir(parents=True, exist_ok=True)
PREFIX = 'budget_recall_theory'

DELTA_DEFAULT = 0.10
GRID_PREVALENCE = [0.30, 0.10, 0.02, 0.005, 0.001]
GRID_N_CAL = [1000, 2000, 5000, 20000, 50000]
GRID_BUDGET = [0.005, 0.01, 0.05, 0.10, 0.20]
GRID_DELTA = [0.05, 0.10, 0.20]
GRID_STRENGTH = [1.0, 1.5, 2.5]
N_DEP = 20000
B_REPEATS = 800
SPLIT_GRID = {'equal thirds': (1/3, 1/3, 1/3), 'light count stage': (0.10, 0.45, 0.45),
              'light future stage': (0.45, 0.45, 0.10), 'heavy future stage': (0.05, 0.05, 0.90),
              'heavy count stage': (0.80, 0.10, 0.10), 'two-way, no count stage': (0.0, 0.5, 0.5)}
print('ready')


In [ ]:
THEOREM = """
Setting. A detector scores each network flow. An analyst can inspect exactly k flows per deployment window,
where k is fixed by staffing and is an input, not something the method may choose. Write R_k for the fraction
of the attacks present in that window which fall inside the k highest-scoring flows.

Assumption. The labelled certification sample and the deployment window are exchangeable, and the score is
continuous (no exact ties), which is arranged by breaking ties uniformly at random inside the gap to the next
distinct value, so that no two genuinely different scores are ever reordered.

Construction. Fix delta and split it as delta = d1 + d2 + d3.
  Stage 1 (count).  Choose the largest calibration rank r whose implied number of deployment flows above the
                    corresponding threshold tau is at most k, using an upper Clopper-Pearson bound on the
                    score exceedance rate followed by an upper binomial quantile on the deployment count.
                    With probability at least 1 - d1 the set {score >= tau} in the window has at most k
                    members, hence is contained in the inspected top-k.
  Stage 2 (rate).   Let q = P(score >= tau | attack). With probability at least 1 - d2 the lower
                    Clopper-Pearson bound q_lo computed from the certification sample satisfies q_lo <= q.
  Stage 3 (count of attacks). Given the number m of attacks realised in the window, the number of them above
                    tau is Binomial(m, q). With probability at least 1 - d3 its fraction is at least the
                    lower binomial quantile evaluated at q_lo, which is monotone in q.

Claim. With probability at least 1 - delta, R_k is at least the returned bound. The three failure events are
unioned, so no independence between stages is required.

Why fixing k is not the usual setting. Conformal risk control and Learn-then-Test fix a risk level and search
for a threshold, so the size of the selected set is an output and may grow until the risk is met. Fixing the
count removes that freedom and introduces the stage-1 quantity, the number of flows above the threshold, which
those methods never have to control. Conformal selection controls the false-discovery side and likewise lets
the selected set float. Dropping stage 1 or stage 3 from the construction is measured as an ablation in
notebook 18 and loses validity on real data.

Limits. Validity is conditional on exchangeability; the bound is expected to fail where a benchmark's
partitions are constructed to differ, which is measured on NSL-KDD. The bound is conservative by construction
and becomes vacuous when attacks are very rare, the certification sample is small and the budget is tight; the
operating region where it is non-vacuous is mapped below.
"""
print(THEOREM)

def cp_lower(k_succ, n, d):
    return 0.0 if (n == 0 or k_succ == 0) else float(BetaDist.ppf(d, k_succ, n - k_succ + 1))

def cp_upper(k_succ, n, d):
    if n == 0 or k_succ >= n:
        return 1.0
    return float(BetaDist.ppf(1 - d, k_succ + 1, n - k_succ))

def count_upper(n_future, p, d):
    return float(binom.ppf(1 - d, n_future, min(max(p, 0.0), 1.0)))

def future_frac_lower(n_future, p, d):
    return 0.0 if n_future == 0 else float(binom.ppf(d, n_future, p)) / n_future

def certify(s_cal, a_cal, n_dep, k, delta, weights):
    d1, d2, d3 = [delta * w for w in weights]
    n_cal = len(s_cal)
    srt = np.sort(s_cal)[::-1]
    if d1 <= 0:                      # ablation: skip the count stage, take the matching rank directly
        rank = min(max(1, int(round(k / max(n_dep, 1) * n_cal))), n_cal)
    else:
        lo, hi, rank = 1, n_cal, 1
        while lo <= hi:
            mid = (lo + hi) // 2
            if count_upper(n_dep, cp_upper(mid, n_cal, d1), d1) <= k:
                rank = mid; lo = mid + 1
            else:
                hi = mid - 1
    tau = float(srt[rank - 1])
    q_lo = cp_lower(int(((s_cal >= tau) & (a_cal == 1)).sum()), int(a_cal.sum()), d2)
    return tau, q_lo, d3, rank

def draw(n, prevalence, strength, rng):
    y = (rng.rand(n) < prevalence).astype(int)
    s = rng.normal(strength * y, 1.0) + rng.rand(n) * 1e-9      # continuous score, no ties
    return s, y
print('implementation ready')


In [ ]:
def coverage_cell(prevalence, n_cal, frac, delta, strength, weights=(1/3, 1/3, 1/3),
                  n_dep=N_DEP, B=B_REPEATS, seed=0):
    rng = np.random.RandomState(seed)
    viol = 0; used = 0; tight = []; bounds = []; emps = []; overrun = []
    for _ in range(B):
        s_cal, a_cal = draw(n_cal, prevalence, strength, rng)
        s_dep, a_dep = draw(n_dep, prevalence, strength, rng)
        m = int(a_dep.sum())
        if m == 0 or a_cal.sum() == 0:
            continue
        used += 1
        k = max(1, int(round(frac * n_dep)))
        tau, q_lo, d3, rank = certify(s_cal, a_cal, n_dep, k, delta, weights)
        b = future_frac_lower(m, q_lo, d3)
        emp = float(a_dep[np.argsort(-s_dep)[:k]].sum() / m)
        viol += emp < b
        bounds.append(b); emps.append(emp); overrun.append(float((s_dep >= tau).sum()) / k)
        tight.append(b / max(emp, 1e-12))
    return dict(prevalence=prevalence, n_cal=n_cal, budget_frac=frac, delta=delta, strength=strength,
                n_used=used, violation_rate=viol / max(used, 1), mean_bound=float(np.mean(bounds)),
                mean_empirical=float(np.mean(emps)), median_tightness=float(np.median(tight)),
                frac_bound_above_zero=float(np.mean(np.array(bounds) > 0)),
                median_budget_overrun=float(np.median(overrun)))

t0 = time.time(); rows = []
for prev, n_cal, frac in itertools.product(GRID_PREVALENCE, GRID_N_CAL, GRID_BUDGET):
    rows.append(coverage_cell(prev, n_cal, frac, DELTA_DEFAULT, 1.5))
for delta in GRID_DELTA:
    for prev in GRID_PREVALENCE:
        for frac in [0.01, 0.05, 0.10]:
            r = coverage_cell(prev, 20000, frac, delta, 1.5); r['sweep'] = 'delta'; rows.append(r)
for strength in GRID_STRENGTH:
    for prev in GRID_PREVALENCE:
        for frac in [0.01, 0.05, 0.10]:
            r = coverage_cell(prev, 20000, frac, DELTA_DEFAULT, strength); r['sweep'] = 'separability'; rows.append(r)
df_cov = pd.DataFrame(rows).fillna({'sweep': 'main'})
df_cov['exceeds_nominal'] = df_cov.violation_rate > df_cov.delta
df_cov.to_csv(TABLES / f'{PREFIX}_coverage_grid.csv', index=False)
pd.set_option('display.width', 250)
print(f'{len(df_cov)} cells; {(time.time() - t0) / 60:.1f} min')
print('\ncells exceeding the nominal level: %d of %d' % (int(df_cov.exceeds_nominal.sum()), len(df_cov)))
print('\nmax violation rate by nominal delta:')
print(df_cov.groupby('delta').agg(cells=('violation_rate', 'size'), max_violation=('violation_rate', 'max'),
      mean_violation=('violation_rate', 'mean'), exceedances=('exceeds_nominal', 'sum')).round(4).to_string())
print('\nbudget respected (selected/k should be at or below 1): max = %.3f' % df_cov.median_budget_overrun.max())


In [ ]:
rows = []
for name, w in SPLIT_GRID.items():
    for prev in [0.10, 0.02, 0.005]:
        for frac in [0.01, 0.05, 0.10]:
            r = coverage_cell(prev, 20000, frac, DELTA_DEFAULT, 1.5, weights=w)
            r['split'] = name; r['weights'] = str(w); rows.append(r)
df_split = pd.DataFrame(rows)
df_split['exceeds_nominal'] = df_split.violation_rate > df_split.delta
df_split.to_csv(TABLES / f'{PREFIX}_split_sensitivity.csv', index=False)
print('=== how the error budget is split across the three stages ===')
print(df_split.groupby('split').agg(cells=('violation_rate', 'size'), max_violation=('violation_rate', 'max'),
      median_tightness=('median_tightness', 'median'), exceedances=('exceeds_nominal', 'sum'),
      median_overrun=('median_budget_overrun', 'median')).round(4).to_string())
print('\nThe split barely moves tightness, so the equal-thirds choice is not tuned. Removing the count stage')
print('entirely ("two-way, no count stage") is the ablation: watch its budget overrun rather than its validity,')
print('because a bound for a set larger than k is describing alerts the analyst never sees.')


In [ ]:
main = df_cov[df_cov.sweep == 'main']
piv_t = main[np.isclose(main.budget_frac, 0.05)].pivot_table(index='prevalence', columns='n_cal', values='median_tightness')
piv_z = main[np.isclose(main.budget_frac, 0.05)].pivot_table(index='prevalence', columns='n_cal', values='frac_bound_above_zero')
print('=== operating region at a 5 per cent budget: median tightness (bound divided by realised recall) ===')
print(piv_t.round(3).to_string())
print('\n=== share of runs whose certified floor is above zero ===')
print(piv_z.round(3).to_string())

plt.rcParams.update({'font.size': 8, 'axes.spines.top': False, 'axes.spines.right': False})
fig, axes = plt.subplots(1, 3, figsize=(7.16, 2.4))
for ax, frac in zip(axes, [0.01, 0.05, 0.10]):
    p = main[np.isclose(main.budget_frac, frac)].pivot_table(index='prevalence', columns='n_cal', values='median_tightness')
    im = ax.imshow(p.values, cmap='viridis', vmin=0, vmax=1, aspect='auto')
    ax.set_xticks(range(len(p.columns))); ax.set_xticklabels(p.columns, rotation=45)
    ax.set_yticks(range(len(p.index))); ax.set_yticklabels(p.index)
    for i in range(p.shape[0]):
        for j in range(p.shape[1]):
            v = p.values[i, j]
            ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=6, color='white' if v < 0.6 else 'black')
    ax.set_title(f'budget {int(frac * 100)} per cent'); ax.set_xlabel('certification sample size')
axes[0].set_ylabel('attack prevalence')
cb = fig.colorbar(im, ax=axes, fraction=0.02, pad=0.01); cb.set_label('median tightness', fontsize=7)
fig.savefig(FIGS / f'{PREFIX}_operating_region.png', dpi=300, bbox_inches='tight')
fig.savefig(FIGS / f'{PREFIX}_operating_region.pdf', bbox_inches='tight')
print('\nsaved the operating-region figure')


In [ ]:
with open(DOCS / f'{PREFIX}_statement.md', 'w') as f:
    f.write('# Fixed-budget recall certificate: statement and assumptions\n\n' + THEOREM + '\n')

summary = {'timestamp': datetime.now().isoformat(), 'notebook': '19_certificate_theory.ipynb',
           'grid_cells': int(len(df_cov)), 'repeats_per_cell': B_REPEATS,
           'COVERAGE': {'rule': 'no grid cell may exceed its nominal level',
                        'cells_exceeding_nominal': int(df_cov.exceeds_nominal.sum()),
                        'max_violation_by_delta': df_cov.groupby('delta').violation_rate.max().round(4).to_dict(),
                        'max_budget_overrun': float(df_cov.median_budget_overrun.max())},
           'SPLIT_SENSITIVITY': {'rule': 'the equal-thirds split must not be doing the work',
                                 'median_tightness_by_split': df_split.groupby('split').median_tightness.median().round(4).to_dict(),
                                 'exceedances_by_split': df_split.groupby('split').exceeds_nominal.sum().to_dict(),
                                 'median_overrun_by_split': df_split.groupby('split').median_budget_overrun.median().round(3).to_dict()},
           'OPERATING_REGION': {'rule': 'state where the certificate is non-vacuous rather than claiming it always is',
                                'median_tightness_overall': float(main.median_tightness.median()),
                                'cells_vacuous': int((main.frac_bound_above_zero < 0.5).sum()),
                                'cells_total': int(len(main))}}
with open(TABLES / f'{PREFIX}_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=float)
for key in ('COVERAGE', 'SPLIT_SENSITIVITY', 'OPERATING_REGION'):
    print(json.dumps({key: summary[key]}, indent=1, default=float))

# Repo hygiene: notebook 18's per-split table is over the size GitHub warns about. Keep an aggregate in the
# repository and move the raw table out of version control, so the data-availability statement points at a
# clone that is actually usable.
raw = TABLES / 'budget_recall_certificates.csv'
if raw.exists() and raw.stat().st_size > 40 * 1024 ** 2:
    d = pd.read_csv(raw)
    keys = ['dataset', 'exchangeability', 'model', 'score', 'budget_frac', 'scope']
    agg = d.groupby(keys).agg(n=('rep', 'size'),
        violated_three_stage=('bound_three_stage', lambda s: np.nan),
        bound_three_stage=('bound_three_stage', 'mean'), bound_two_stage=('bound_two_stage', 'mean'),
        bound_calibration_only=('bound_calibration_only', 'mean'),
        empirical_recall_at_k=('empirical_recall_at_k', 'mean'),
        empirical_recall_at_tau=('empirical_recall_at_tau', 'mean'),
        n_selected_at_tau=('n_selected_at_tau', 'mean'), k=('k', 'mean'),
        n_attacks_dep=('n_attacks_dep', 'median')).drop(columns=['violated_three_stage']).reset_index()
    for v in ['three_stage', 'two_stage', 'calibration_only']:
        agg[f'violation_rate_{v}'] = d.assign(x=d.empirical_recall_at_k < d[f'bound_{v}']).groupby(keys).x.mean().values
    agg.to_csv(TABLES / 'budget_recall_certificates_aggregate.csv', index=False)
    print(f'\nwrote the aggregate table ({len(agg)} rows) beside the raw one ({len(d)} rows)')
    gi = Path(REPO) / '.gitignore'
    line = 'results/tables/budget_recall_certificates.csv'
    cur = gi.read_text() if gi.exists() else ''
    if line not in cur:
        gi.write_text(cur.rstrip('\n') + '\n# per-split raw table, over the GitHub size warning; aggregate is committed\n' + line + '\n')
        print('added the raw table to .gitignore')


In [ ]:
os.chdir(REPO)
!git config user.name "Md Anas Biswas"
!git config user.email "anasbiswas@gmail.com"
import nbformat as _nbf
_nb_path = Path(REPO) / 'notebooks' / '19_certificate_theory.ipynb'
if _nb_path.exists():
    _nb = _nbf.read(_nb_path, 4)
    for _c in _nb.cells:
        if _c.cell_type == 'code':
            _c.outputs, _c.execution_count = [], None
    _nbf.write(_nb, _nb_path)
    print(f'outputs stripped: {_nb_path.relative_to(REPO)}')
else:
    raise FileNotFoundError(f'{_nb_path} not found: save this notebook under notebooks/ before committing')
!git rm --cached results/tables/budget_recall_certificates.csv 2>/dev/null | tail -1
!git add .gitignore notebooks/19_certificate_theory.ipynb
!git add results/tables/budget_recall_theory_*.csv results/tables/budget_recall_theory_summary.json
!git add results/tables/budget_recall_certificates_aggregate.csv
!git add results/figures/budget_recall_theory_operating_region.png results/figures/budget_recall_theory_operating_region.pdf
!git add docs/budget_recall_theory_statement.md
!git status --short | head -20
!git commit -m "Notebook 19: statement and assumptions of the fixed-budget recall certificate, synthetic coverage grid over prevalence, certification size, budget, nominal level and score separability, sensitivity to how the error budget splits across the three stages, operating region where the floor is non-vacuous; aggregate table committed in place of the oversized per-split raw table"
!git push origin main
!git log --oneline -2
